# INF0415 — Semana 9: Algoritmos Meméticos e Warm-Start
## Competição Algorithm Discovery
**Bacharelado em Inteligência Artificial — UFG / Instituto de Informática**

---

| Parte | Conteúdo | Atividade |
|-------|----------|-----------|
| 1 | GA Memético (GA + 2-opt) | Implementar 2-opt e medir ganho sobre GA puro |
| 2 | Warm-Start | Comparar cold vs. warm-start com gráficos |
| 3 | Competição Algorithm Discovery | Configuração livre, 5 sementes fixas, ranking da turma |

**Pré-requisito:** Semanas 7 (GA), 8 (ACO). Dataset TSP-30: seed=42, n=30 cidades.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time, copy
from itertools import combinations

np.random.seed(0)

plt.rcParams.update({
    'figure.facecolor': '#0D1117', 'axes.facecolor': '#161B27',
    'axes.edgecolor': '#30363D', 'axes.labelcolor': '#C9D1D9',
    'xtick.color': '#8B949E', 'ytick.color': '#8B949E',
    'text.color': '#C9D1D9', 'grid.color': '#21262D',
    'grid.linestyle': '--', 'grid.alpha': 0.5,
    'lines.linewidth': 2, 'font.size': 12,
})
print("Imports OK.")


In [ ]:
# ── Dataset TSP-30 (idêntico às semanas anteriores) ──────────────────────
np.random.seed(42)
CIDADES   = np.random.rand(30, 2) * 100
N_CIDADES = len(CIDADES)

def dist_matrix(cidades):
    n = len(cidades)
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            D[i, j] = np.linalg.norm(cidades[i] - cidades[j])
    return D

DIST = dist_matrix(CIDADES)

def custo_rota(rota, D=DIST):
    total  = sum(D[rota[i], rota[i+1]] for i in range(len(rota)-1))
    total += D[rota[-1], rota[0]]
    return total

def vizinho_mais_proximo(inicio, D=DIST):
    n = len(D); rota = [inicio]; vis = {inicio}
    for _ in range(n-1):
        c = rota[-1]
        prox = min((D[c,j],j) for j in range(n) if j not in vis)
        rota.append(prox[1]); vis.add(prox[1])
    return rota

# Resultados validados (semanas anteriores)
HISTORICO = {
    'Aleatório':       (1564.55, 119.95),
    'Vizinho Próximo': (562.11,   0.00),
    'SA':              (475.40,  18.60),
    'Busca Tabu':      (461.20,   3.70),
    'GA Config B':     (461.80,   1.70),
    'ACO':             (455.06,   0.01),
}

custo_nn_ref = vizinho_mais_proximo(0)
print(f"TSP-30 pronto. Custo NN (cidade 0): {custo_rota(custo_nn_ref):.2f}")
print(f"Histórico: {list(HISTORICO.keys())}")


In [ ]:
# ── GA Base (GA Config B — reutilizado da Semana 7) ─────────────────────
def torneio(pop, vals, k=3):
    idxs = np.random.choice(len(pop), k, replace=False)
    return pop[idxs[np.argmin(vals[idxs])]]

def ox_crossover(p1, p2):
    n = len(p1)
    a, b = sorted(np.random.choice(n, 2, replace=False))
    filho = [-1] * n
    filho[a:b+1] = p1[a:b+1]
    seg = [x for x in p2 if x not in filho[a:b+1]]
    pos = [(i % n) for i in range(b+1, b+1+n) if i % n < a or i % n > b]
    for p, v in zip(pos, seg):
        filho[p] = v
    return np.array(filho)

def mutacao_inversao(rota):
    n = len(rota); r = rota.copy()
    i, j = sorted(np.random.choice(n, 2, replace=False))
    r[i:j+1] = r[i:j+1][::-1]
    return r

def ga_base(D=DIST, pop_size=100, n_gen=200,
            tx_cross=0.85, tx_mut=0.15, elite=2,
            seed=42, init_pop=None):
    """
    GA Config B: seleção por torneio, OX, mutação por inversão, elitismo.
    Se init_pop for fornecida, usa-a como população inicial (warm-start).
    Retorna: melhor_rota, melhor_custo, curva_convergência
    """
    np.random.seed(seed)
    n = len(D)
    if init_pop is not None:
        pop = np.array([np.array(r) for r in init_pop[:pop_size]])
    else:
        pop = np.array([np.random.permutation(n) for _ in range(pop_size)])
    vals = np.array([custo_rota(r, D) for r in pop])
    melhor_idx  = np.argmin(vals)
    melhor_rota = pop[melhor_idx].copy()
    melhor_custo = vals[melhor_idx]
    curva = [melhor_custo]
    for _ in range(n_gen):
        nova_pop = list(pop[np.argsort(vals)[:elite]])
        while len(nova_pop) < pop_size:
            p1 = torneio(pop, vals)
            p2 = torneio(pop, vals)
            filho = ox_crossover(p1, p2) if np.random.rand() < tx_cross else p1.copy()
            if np.random.rand() < tx_mut:
                filho = mutacao_inversao(filho)
            nova_pop.append(filho)
        pop  = np.array(nova_pop)
        vals = np.array([custo_rota(r, D) for r in pop])
        idx  = np.argmin(vals)
        if vals[idx] < melhor_custo:
            melhor_custo = vals[idx]
            melhor_rota  = pop[idx].copy()
        curva.append(melhor_custo)
    return melhor_rota, melhor_custo, curva

print("GA Base (Config B) definido.")


---
## Parte 1 — GA Memético (GA + 2-opt)

### O que é um algoritmo memético?

> "Um indivíduo herda genes dos pais **e** aprende com a experiência local."
> — Pablo Moscato, 1989

O GA Memético adiciona um passo de **busca local** após o cruzamento e mutação.
Para o TSP, usamos o **operador 2-opt**: remove dois arcos e reconecta de forma mais curta.

### A equação do ganho

```
Custo final = GA puro         →  ~462
             + 2-opt (local)  →  ~430–445  (estimado)
```

### O operador 2-opt

Dado um tour `[..., A, B, C, D, E, ...]`, o 2-opt:
1. Remove os arcos `(A→B)` e `(D→E)`
2. Inverte o segmento entre eles: `[..., A, D, C, B, E, ...]`
3. Aceita se `d(A,D) + d(B,E) < d(A,B) + d(D,E)`

Repete até não encontrar mais melhorias (**ótimo 2-opt local**).

**Complexidade:** O(n²) por passagem · Para n=30: ~870 comparações por indivíduo


In [ ]:
# ── TODO: implemente o operador 2-opt ────────────────────────────────────

def dois_opt(rota, D=DIST, max_iter=None):
    """
    Aplica o operador 2-opt a uma rota até convergir (ótimo local 2-opt).

    Pseudocódigo:
        melhorou = True
        enquanto melhorou:
            melhorou = False
            para i em 0..n-2:
                para j em i+2..n:
                    nova_rota = rota[:i+1] + reverso(rota[i+1:j+1]) + rota[j+1:]
                    se custo(nova_rota) < custo(rota):
                        rota = nova_rota
                        melhorou = True
        retornar rota

    Parâmetros:
        rota     : lista ou array de índices (permutação de cidades)
        D        : matriz de distâncias
        max_iter : limite de iterações (None = até convergir)

    Retorna:
        rota melhorada (numpy array)
    """
    # TODO: converta rota para lista mutável
    rota = list(rota)
    n = len(rota)
    it = 0

    # TODO: loop externo "enquanto melhorou"
    melhorou = True
    while melhorou:
        melhorou = False

        # TODO: loop duplo sobre pares (i, j)
        for i in range(n - 1):
            for j in range(i + 2, n):

                # TODO: calcular ganho sem montar a rota inteira:
                # ganho = D[rota[i], rota[j]] + D[rota[i+1], rota[(j+1)%n]]
                #       - D[rota[i], rota[i+1]] - D[rota[j], rota[(j+1)%n]]
                # Dica: se ganho < -1e-10, a troca melhora
                ganho = None   # <-- substitua

                if ganho is not None and ganho < -1e-10:
                    # TODO: reverter o segmento rota[i+1 : j+1]
                    rota[i+1:j+1] = None  # <-- substitua
                    melhorou = True

        it += 1
        if max_iter and it >= max_iter:
            break

    return np.array(rota)


In [ ]:
# ── Verificar 2-opt ──────────────────────────────────────────────────────
rota_aleatoria = np.random.permutation(N_CIDADES)
custo_antes = custo_rota(rota_aleatoria)
rota_melhorada = dois_opt(rota_aleatoria)
custo_depois = custo_rota(rota_melhorada)

print(f"Custo antes  do 2-opt: {custo_antes:.2f}")
print(f"Custo depois do 2-opt: {custo_depois:.2f}")
print(f"Melhora: {custo_antes - custo_depois:.2f} ({(custo_antes-custo_depois)/custo_antes*100:.1f}%)")
assert custo_depois <= custo_antes + 1e-9, "2-opt não pode piorar a rota!"
print("✓ 2-opt correto.")


### 1.2 Integrando 2-opt ao GA

O GA Memético adiciona **uma chamada a `dois_opt`** após o cruzamento e mutação de cada filho.
Isso é a versão **Lamarckiana**: o filho melhorado substitui o original na população.

**Pseudocódigo do loop interno:**
```
para cada nova geração:
    para cada par de pais selecionados:
        filho = OX(p1, p2)
        filho = mutação_inversão(filho)  se prob < tx_mut
        filho = dois_opt(filho)          ← NOVO
    nova_pop.append(filho)
```


In [ ]:
# ── TODO: implemente o GA Memético ──────────────────────────────────────

def ga_memetico(D=DIST, pop_size=100, n_gen=200,
                tx_cross=0.85, tx_mut=0.15, elite=2,
                seed=42, init_pop=None, aplicar_2opt=True):
    """
    GA Memético: GA Config B + 2-opt em cada filho.

    Parâmetros adicionais vs. ga_base:
        aplicar_2opt : se True, aplica dois_opt em cada filho gerado.
        init_pop     : população inicial opcional (warm-start).

    Retorna: melhor_rota, melhor_custo, curva_convergência
    """
    np.random.seed(seed)
    n = len(D)

    # TODO: inicializar população (usar init_pop se fornecida, senão aleatório)
    if init_pop is not None:
        pop = np.array([np.array(r) for r in init_pop[:pop_size]])
    else:
        pop = None  # <-- substitua por inicialização aleatória

    # TODO: avaliar fitness inicial
    vals = None  # shape (pop_size,)

    # TODO: identificar melhor solução inicial
    melhor_rota  = None
    melhor_custo = None
    curva = [melhor_custo]

    for _ in range(n_gen):
        # Elitismo: manter os `elite` melhores
        nova_pop = list(pop[np.argsort(vals)[:elite]])

        while len(nova_pop) < pop_size:
            # TODO: seleção por torneio (use a função `torneio`)
            p1 = None
            p2 = None

            # TODO: cruzamento OX (use ox_crossover)
            filho = None

            # TODO: mutação por inversão (use mutacao_inversao)
            if np.random.rand() < tx_mut:
                filho = None  # substitua

            # TODO: aplicar 2-opt se aplicar_2opt for True
            if aplicar_2opt:
                filho = None  # substitua

            nova_pop.append(filho)

        # TODO: atualizar pop e vals para a nova geração
        pop  = None
        vals = None

        # TODO: atualizar melhor solução global
        idx = None
        pass

        curva.append(melhor_custo)

    return melhor_rota, melhor_custo, curva


In [ ]:
# ── Executar GA Memético — 5 sementes ────────────────────────────────────
SEMENTES = [42, 123, 456, 789, 2024]

# GA puro (referência)
print("Rodando GA puro (Semana 7)...")
res_ga_puro = []
curvas_ga_puro = []
t0 = time.time()
for s in SEMENTES:
    _, custo, curva = ga_base(seed=s)
    res_ga_puro.append(custo)
    curvas_ga_puro.append((s, curva))
t_puro = time.time() - t0
media_puro, dp_puro = np.mean(res_ga_puro), np.std(res_ga_puro)

# GA Memético (com 2-opt)
print("Rodando GA Memético (com 2-opt)...")
res_mem = []
curvas_mem = []
t0 = time.time()
for s in SEMENTES:
    _, custo, curva = ga_memetico(seed=s)
    res_mem.append(custo)
    curvas_mem.append((s, curva))
t_mem = time.time() - t0
media_mem, dp_mem = np.mean(res_mem), np.std(res_mem)

print(f"\nGA Puro:     {media_puro:.2f} ± {dp_puro:.2f}  ({t_puro:.1f}s)")
print(f"GA Memético: {media_mem:.2f} ± {dp_mem:.2f}  ({t_mem:.1f}s)")
print(f"Ganho: {media_puro - media_mem:.2f}  ({(media_puro-media_mem)/media_puro*100:.1f}%)")


In [ ]:
# ── Convergência: GA Puro vs. GA Memético ────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

for (s, c), cor in zip(curvas_ga_puro, ['#8B949E']*5):
    ax1.plot(c, color=cor, alpha=0.7, linewidth=1.5, label=f'seed={s}' if s==42 else None)
for (s, c), cor in zip(curvas_mem, ['#2FBE8F']*5):
    ax1.plot(c, color=cor, alpha=0.85, linewidth=1.5, linestyle='--')
from matplotlib.lines import Line2D
handles = [Line2D([0],[0],color='#8B949E',label='GA Puro'),
           Line2D([0],[0],color='#2FBE8F',linestyle='--',label='GA Memético')]
ax1.legend(handles=handles, facecolor='#161B27', edgecolor='#30363D')
ax1.set_title('Convergência: GA Puro vs. Memético'); ax1.set_xlabel('Geração'); ax1.set_ylabel('Melhor custo')
ax1.grid(True)

# Bar chart comparativo
algos_comp = list(HISTORICO.keys()) + ['GA Memético']
medias_comp = [HISTORICO[a][0] for a in HISTORICO] + [media_mem]
dps_comp    = [HISTORICO[a][1] for a in HISTORICO] + [dp_mem]
cores_comp  = ['#8B949E']*len(HISTORICO) + ['#2FBE8F']
bars = ax2.bar(algos_comp, medias_comp, color=cores_comp, alpha=0.85,
               yerr=dps_comp, capsize=4, error_kw={'ecolor':'white','linewidth':1.2})
ax2.set_ylabel('Custo médio'); ax2.set_title('Ranking — TSP-30')
ax2.tick_params(axis='x', rotation=25)
ax2.set_ylim(400, 1000)
ax2.grid(True, axis='y')
for bar, val in zip(bars, medias_comp):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+8, f'{val:.0f}',
             ha='center', va='bottom', fontsize=9, color='white')
plt.tight_layout()
plt.savefig('p1_ga_vs_memetico.png', dpi=110, bbox_inches='tight', facecolor='#0D1117')
plt.show()
print(f"\nGA Memético:  {media_mem:.2f} ± {dp_mem:.2f}")
print(f"ACO (Sem. 8): 455.06 ± 0.01")


---
## Parte 2 — Warm-Start

### O problema do arranque frio

Um GA com inicialização aleatória começa com custo médio ~1564.
Ele precisa de muitas gerações só para descer ao nível onde soluções razoáveis aparecem.

### Estratégia: Vizinho Mais Próximo Diverso

Gerar `pop_size` soluções NN, cada uma começando de uma cidade diferente:
- Todas têm custo ~562 (muito melhor que aleatório)
- Cada uma é diferente (diversidade preservada)
- O GA refina a partir daí, em vez de explorar do zero

### Resultado esperado

| Métrica | Cold-Start | Warm-Start NN |
|---------|-----------|--------------|
| Custo geração 0 | ~1564 | ~562 |
| Custo geração 50 | ~490 | ~450 |
| Custo final | ~430–445 | **~420–435** *(esperado)* |
| Gerações para < 480 | ~80 | ~10 |


### 2.1 Gerador de Warm-Start


In [ ]:
# ── TODO: implementar gerador de população warm-start ────────────────────

def gerar_populacao_warm(pop_size, D=DIST, fracao_nn=1.0):
    """
    Gera uma população inicial com warm-start via Vizinho Mais Próximo.

    Parâmetros:
        pop_size   : tamanho da população a gerar
        D          : matriz de distâncias
        fracao_nn  : fração da pop com NN (restante é aleatória, para diversidade)
                     fracao_nn=1.0 → todos NN | fracao_nn=0.5 → híbrido 50/50

    Retorna:
        pop : lista de listas (cada item é uma rota)
    """
    n = len(D)
    pop = []

    n_nn  = int(pop_size * fracao_nn)
    n_rnd = pop_size - n_nn

    # TODO: gerar n_nn soluções NN a partir de cidades de início diversas
    # Dica: use vizinho_mais_proximo(inicio, D) variando `inicio` de 0 a n-1
    # Se n_nn > n_cidades, recicle as cidades (use modulo %)
    for k in range(n_nn):
        inicio = None   # <-- escolha a cidade de início
        rota   = None   # <-- chame vizinho_mais_proximo
        pop.append(rota)

    # TODO: gerar n_rnd soluções aleatórias
    for _ in range(n_rnd):
        pop.append(None)  # <-- substitua por np.random.permutation(n).tolist()

    return pop


In [ ]:
# ── Comparar cold-start vs. warm-start ───────────────────────────────────
pop_warm = gerar_populacao_warm(100, fracao_nn=1.0)
custo_inicial_warm = np.mean([custo_rota(r) for r in pop_warm])
custo_inicial_cold = np.mean([custo_rota(np.random.permutation(N_CIDADES)) for _ in range(100)])
print(f"Custo médio inicial — cold: {custo_inicial_cold:.1f}  |  warm: {custo_inicial_warm:.1f}")

print("\nRodando GA Memético — 5 sementes — cold-start e warm-start...")
res_cold, curvas_cold = [], []
res_warm, curvas_warm = [], []

for s in SEMENTES:
    _, custo, curva = ga_memetico(seed=s)
    res_cold.append(custo); curvas_cold.append((s, curva))

    np.random.seed(s)  # reset antes de gerar warm-pop
    pop_w = gerar_populacao_warm(100, fracao_nn=1.0)
    _, custo, curva = ga_memetico(seed=s, init_pop=pop_w)
    res_warm.append(custo); curvas_warm.append((s, curva))

media_cold, dp_cold = np.mean(res_cold), np.std(res_cold)
media_warm, dp_warm = np.mean(res_warm), np.std(res_warm)
print(f"\nGA Memético cold-start: {media_cold:.2f} ± {dp_cold:.2f}")
print(f"GA Memético warm-start: {media_warm:.2f} ± {dp_warm:.2f}")
print(f"Ganho do warm-start:    {media_cold - media_warm:.2f}")


In [ ]:
# ── Gráficos: convergência cold vs. warm ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
for (s, c) in curvas_cold:
    ax.plot(c, color='#E85D40', alpha=0.7, linewidth=1.5)
for (s, c) in curvas_warm:
    ax.plot(c, color='#2FBE8F', alpha=0.85, linewidth=1.5, linestyle='--')
from matplotlib.lines import Line2D
handles = [Line2D([0],[0],color='#E85D40',label='Cold-start'),
           Line2D([0],[0],color='#2FBE8F',linestyle='--',label='Warm-start NN')]
ax.legend(handles=handles, facecolor='#161B27', edgecolor='#30363D')
ax.set_title('GA Memético — Cold vs. Warm'); ax.set_xlabel('Geração'); ax.set_ylabel('Melhor custo')
ax.axhline(480, color='#F4C842', linewidth=1, linestyle=':', label='480')
ax.grid(True)

ax2 = axes[1]
# Gerações necessárias para atingir custo < 480
def gen_para_threshold(curva, th=480):
    for i, v in enumerate(curva):
        if v < th: return i
    return len(curva)

gens_cold = [gen_para_threshold(c) for _, c in curvas_cold]
gens_warm = [gen_para_threshold(c) for _, c in curvas_warm]
x = np.arange(len(SEMENTES))
w = 0.35
ax2.bar(x - w/2, gens_cold, w, color='#E85D40', alpha=0.85, label='Cold-start')
ax2.bar(x + w/2, gens_warm, w, color='#2FBE8F', alpha=0.85, label='Warm-start')
ax2.set_xticks(x); ax2.set_xticklabels([f'seed={s}' for s in SEMENTES], rotation=15)
ax2.set_ylabel('Geração'); ax2.set_title('Gerações até custo < 480')
ax2.legend(facecolor='#161B27', edgecolor='#30363D'); ax2.grid(True, axis='y')
plt.tight_layout()
plt.savefig('p2_cold_vs_warm.png', dpi=110, bbox_inches='tight', facecolor='#0D1117')
plt.show()


In [ ]:
# ── Tabela resumo Partes 1 e 2 ───────────────────────────────────────────
print("=" * 62)
print(f"  {'Algoritmo':<28} {'Média':>10} {'DP':>10}")
print("-" * 62)
for alg, (med, dp) in HISTORICO.items():
    print(f"  {alg:<28} {med:>10.2f} {dp:>10.2f}")
print(f"  {'GA Memético (cold)':<28} {media_cold:>10.2f} {dp_cold:>10.2f}")
print(f"  {'GA Memético (warm)':<28} {media_warm:>10.2f} {dp_warm:>10.2f}")
print("=" * 62)


---
## Parte 3 — Competição Algorithm Discovery

### Sementes fixas para toda a turma
As sementes são **fixas e iguais** para todos: `[42, 123, 456, 789, 2024]`.
Isso garante comparabilidade — ninguém pode fazer overfitting em uma semente.

### Critérios de avaliação
| Critério | Peso | Fórmula |
|----------|------|---------|
| Custo médio | 60% | menor = melhor |
| Desvio padrão | 20% | menor = mais confiável |
| Justificativa oral | 20% | argumentação + evidências |

### Sua missão
Encontrar a melhor configuração dentro de ≤ 60s de execução.
Preencha a célula de configuração abaixo, rode, e copie o resultado
para a planilha compartilhada da turma.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# COMPETIÇÃO — CONFIGURAÇÃO DA DUPLA
# Edite os parâmetros abaixo livremente.
# Regra: execução total ≤ 60 segundos.
# ══════════════════════════════════════════════════════════════════════════

SEMENTES_COMPETICAO = [42, 123, 456, 789, 2024]  # NÃO ALTERAR

# ── Escolha o algoritmo base ──────────────────────────────────────────────
# Opções: 'ga_puro' | 'ga_memetico' | 'hibrido_livre'
ALGORITMO = 'ga_memetico'

# ── Parâmetros do GA (se usar GA ou GA Memético) ─────────────────────────
POP_SIZE  = 100
N_GEN     = 200
TX_CROSS  = 0.85
TX_MUT    = 0.15
ELITE     = 2
APLICAR_2OPT = True

# ── Inicialização ─────────────────────────────────────────────────────────
# Opções: 'cold' | 'warm_nn' | 'warm_hibrido'
INICIALIZACAO  = 'warm_nn'
FRACAO_NN      = 1.0   # usado apenas se INICIALIZACAO == 'warm_hibrido'

print("Configuração da dupla:")
print(f"  Algoritmo:     {ALGORITMO}")
print(f"  Inicialização: {INICIALIZACAO}")
print(f"  Pop: {POP_SIZE}  Ger: {N_GEN}  Cross: {TX_CROSS}  Mut: {TX_MUT}  Elite: {ELITE}  2-opt: {APLICAR_2OPT}")


In [ ]:
# ── Executar competição ───────────────────────────────────────────────────
resultados_comp = []
t0 = time.time()

for seed in SEMENTES_COMPETICAO:
    # Montar inicialização
    if INICIALIZACAO == 'cold':
        init_pop = None
    elif INICIALIZACAO == 'warm_nn':
        np.random.seed(seed)
        init_pop = gerar_populacao_warm(POP_SIZE, fracao_nn=1.0)
    else:  # warm_hibrido
        np.random.seed(seed)
        init_pop = gerar_populacao_warm(POP_SIZE, fracao_nn=FRACAO_NN)

    # Rodar algoritmo
    if ALGORITMO == 'ga_puro':
        _, custo, _ = ga_base(pop_size=POP_SIZE, n_gen=N_GEN,
                              tx_cross=TX_CROSS, tx_mut=TX_MUT,
                              elite=ELITE, seed=seed, init_pop=init_pop)
    elif ALGORITMO == 'ga_memetico':
        _, custo, _ = ga_memetico(pop_size=POP_SIZE, n_gen=N_GEN,
                                  tx_cross=TX_CROSS, tx_mut=TX_MUT,
                                  elite=ELITE, seed=seed, init_pop=init_pop,
                                  aplicar_2opt=APLICAR_2OPT)
    else:
        raise ValueError("Implemente seu algoritmo híbrido livre aqui!")

    resultados_comp.append(custo)
    print(f"  seed={seed:4d}  custo={custo:.2f}")

t_total = time.time() - t0
media_comp = np.mean(resultados_comp)
dp_comp    = np.std(resultados_comp)

print(f"\n{'='*50}")
print(f"  RESULTADO FINAL DA DUPLA")
print(f"  Algoritmo: {ALGORITMO}  |  Init: {INICIALIZACAO}")
print(f"  Média:  {media_comp:.2f}")
print(f"  DP:     {dp_comp:.2f}")
print(f"  Tempo:  {t_total:.1f}s")
print(f"{'='*50}")
print("\n⬆ Copie esses valores para a planilha da turma!")


### Justificativa da dupla

**Escreva aqui (3–5 linhas) por que vocês escolheram essa configuração:**

*(Dica: cite os experimentos das Partes 1 e 2 que embasaram a escolha.
O que vocês aprenderam sobre 2-opt e warm-start que levou a essa decisão?)*


### Questões analíticas Q1–Q4


**Q1.** O GA Memético convergiu mais rápido ou mais lento que o GA puro?
O custo final foi melhor? Explique o trade-off exploração/explotação.

*(Responda aqui com base nos seus resultados da Parte 1)*


**Q2.** Qual foi o impacto do warm-start no número de gerações para atingir custo < 480?
A vantagem persiste até o final?

*(Responda aqui com base nos seus resultados da Parte 2)*


**Q3.** Na competição, qual foi a configuração vencedora da turma?
Ela generalizaria para TSP com 500 cidades?

*(Responda após o resultado da competição — escreva o vencedor e sua análise)*


**Q4.** Se você tivesse que escolher UM algoritmo para TSP com 1000 cidades e
orçamento de 5 minutos, qual escolheria? Justifique com base em complexidade e qualidade.

*(Responda aqui)*


In [ ]:
# ── Ranking final + gráfico completo ─────────────────────────────────────
ranking_completo = dict(HISTORICO)
ranking_completo['GA Memético (cold)'] = (media_cold, dp_cold)
ranking_completo['GA Memético (warm)'] = (media_warm, dp_warm)
ranking_completo['Sua config (comp.)'] = (media_comp, dp_comp)

# Ordenar por custo médio
ranking_ord = dict(sorted(ranking_completo.items(), key=lambda x: x[1][0]))

fig, ax = plt.subplots(figsize=(12, 5))
nomes = list(ranking_ord.keys())
medias2 = [ranking_ord[a][0] for a in nomes]
dps2    = [ranking_ord[a][1] for a in nomes]
cores2  = ['#F4C842' if 'comp' in a else '#2FBE8F' if 'warm' in a else '#8B949E' for a in nomes]
bars = ax.bar(nomes, medias2, color=cores2, alpha=0.85,
              yerr=dps2, capsize=4, error_kw={'ecolor':'white','linewidth':1.2})
ax.set_title('Ranking Final — TSP-30  ·  Semana 9', pad=8)
ax.set_ylabel('Custo médio'); ax.tick_params(axis='x', rotation=30)
ax.set_ylim(390, max(medias2)*1.08)
ax.grid(True, axis='y')
for bar, val in zip(bars, medias2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5, f'{val:.1f}',
            ha='center', va='bottom', fontsize=9, color='white')
plt.tight_layout()
plt.savefig('ranking_final.png', dpi=110, bbox_inches='tight', facecolor='#0D1117')
plt.show()

print("\n🏁 Semana 9 concluída!")
print("Próxima aula: Modelagem Multiobjetivo e Fronteira de Pareto.")
